# Notebook 07 - Full-Graph GCN baseline and transductive stress test

**Tasks 3.1-3.3 (TODO):** build a 2-layer GCN baseline with `GCNConv` wrapped by `to_hetero`, train it on the full graph for a fairer comparison against GraphSAGE, and log the transductive limitations of raw `GCNConv` on bipartite hetero graphs.

**What this notebook does:**
1. Reuses the same processed artifacts as notebook 06 (`hetero_data.pt`, mappings, hidden-user parquet, feature tensors).
2. Runs a smoke-test with **native** `GCNConv + to_hetero` and records the raw bipartite failure expected by TODO 3.3.
3. Trains an **adapted** full-graph GCN that still uses `GCNConv`, `to_hetero`, and the same edge decoder interface as notebook 06.
4. Evaluates the trained model on seen-user and hidden-user full-graph splits, then stores all reports in `data/processed/gcn_link_pred.pt`.

In [1]:
import copy
import json
import random
import traceback
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import average_precision_score, roc_auc_score
from torch_geometric.data import HeteroData
from torch_geometric.nn import GCNConv, to_hetero
from torch_geometric.transforms import RandomLinkSplit
from torch_geometric.utils import add_self_loops, negative_sampling

PROCESSED_DIR = Path('../data/processed/')
CKPT_PATH = PROCESSED_DIR / 'gcn_link_pred.pt'
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
print('torch:', torch.__version__)


class DotProductDecoder(nn.Module):
    def forward(self, user_emb: torch.Tensor, item_emb: torch.Tensor) -> torch.Tensor:
        return (user_emb * item_emb).sum(dim=-1)


class NativeGCNBase(nn.Module):
    """Raw GCNConv stack used only for limitation smoke-tests."""

    def __init__(self, hidden_channels: int, out_channels: int, dropout: float = 0.1):
        super().__init__()
        self.conv1 = GCNConv(-1, hidden_channels, add_self_loops=False, normalize=True)
        self.conv2 = GCNConv(hidden_channels, out_channels, add_self_loops=False, normalize=True)
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.conv2(x, edge_index)
        return x


class BipartiteGCNConv(GCNConv):
    """GCNConv subclass that applies symmetric normalization on bipartite edges."""

    def forward(self, x, edge_index, edge_weight=None):
        if isinstance(x, tuple):
            x_src, x_dst = x
            num_src = x_src.size(0)
            x_all = torch.cat([x_src, x_dst], dim=0)
            edge_index = torch.stack([edge_index[0], edge_index[1] + num_src], dim=0)
            rev_edge_index = torch.stack([edge_index[1], edge_index[0]], dim=0)
            edge_index = torch.cat([edge_index, rev_edge_index], dim=1)
            edge_index, _ = add_self_loops(edge_index, num_nodes=x_all.size(0))
            out_all = super().forward(x_all, edge_index, edge_weight=None)
            return out_all[num_src:]
        return super().forward(x, edge_index, edge_weight=edge_weight)


class AdaptedGCNBase(nn.Module):
    """Runnable 2-layer GCN baseline that keeps the notebook-06 interface."""

    def __init__(self, hidden_channels: int, out_channels: int, dropout: float = 0.1):
        super().__init__()
        self.conv1 = BipartiteGCNConv(-1, hidden_channels, add_self_loops=False, normalize=True)
        self.conv2 = BipartiteGCNConv(hidden_channels, out_channels, add_self_loops=False, normalize=True)
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.conv2(x, edge_index)
        return x


def build_native_gcn_encoder(metadata, hidden_channels: int, out_channels: int, dropout: float = 0.1):
    return to_hetero(NativeGCNBase(hidden_channels, out_channels, dropout), metadata, aggr='sum')


def build_adapted_gcn_encoder(metadata, hidden_channels: int, out_channels: int, dropout: float = 0.1):
    return to_hetero(AdaptedGCNBase(hidden_channels, out_channels, dropout), metadata, aggr='sum')


def make_item_graph_undirected(data: HeteroData) -> HeteroData:
    out = data.clone()
    item_item = out['item', 'also_bought', 'item'].edge_index
    out['item', 'also_bought', 'item'].edge_index = torch.cat([item_item, item_item.flip(0)], dim=1)
    return out


def add_reverse_reviews(data: HeteroData) -> HeteroData:
    out = data.clone()
    out['item', 'rev_reviews', 'user'].edge_index = out['user', 'reviews', 'item'].edge_index.flip(0)
    return out


def sample_edge_labels(pos_edge_index, full_positive_edge_index, num_users, num_items, neg_ratio: float = 1.0):
    pos_edge_index = pos_edge_index.cpu()
    full_positive_edge_index = full_positive_edge_index.cpu()
    pos_labels = torch.ones(pos_edge_index.size(1), dtype=torch.float)
    num_neg = max(int(pos_edge_index.size(1) * neg_ratio), 1)
    neg_edge_index = negative_sampling(
        full_positive_edge_index,
        num_nodes=(num_users, num_items),
        num_neg_samples=num_neg,
        method='sparse',
    )
    edge_label_index = torch.cat([pos_edge_index, neg_edge_index], dim=1)
    edge_label = torch.cat([pos_labels, torch.zeros(neg_edge_index.size(1), dtype=torch.float)], dim=0)
    return edge_label_index, edge_label


def decode_link_logits(z_dict, edge_label_index: torch.Tensor, decoder: nn.Module) -> torch.Tensor:
    z_user = z_dict['user'][edge_label_index[0]]
    z_item = z_dict['item'][edge_label_index[1]]
    return decoder(z_user, z_item)


def capture_error(exc: Exception) -> dict:
    return {
        'type': type(exc).__name__,
        'message': str(exc),
        'traceback_tail': traceback.format_exc().splitlines()[-10:],
    }


def attempt_stage(stage_name: str, fn):
    try:
        value = fn()
        return {'stage': stage_name, 'ok': True, 'value': value}
    except Exception as exc:
        return {'stage': stage_name, 'ok': False, 'error': capture_error(exc)}


def print_stage_result(result: dict):
    if result.get('ok'):
        value = result.get('value', {})
        print(f"[{result['stage']}] ok")
        if isinstance(value, dict):
            for key in ['user_embedding_shape', 'item_embedding_shape', 'logits_shape', 'auroc', 'ap', 'acc_0.5']:
                if key in value:
                    print(f"  {key}: {value[key]}")
    else:
        error = result.get('error', {})
        print(f"[{result['stage']}] failed: {error.get('type')} - {error.get('message')}")


@torch.no_grad()
def eval_fullgraph(encoder: nn.Module, decoder: nn.Module, data_obj: HeteroData, pos_edge_index: torch.Tensor, full_positive_edge_index: torch.Tensor, device: torch.device) -> dict:
    encoder.eval()
    decoder.eval()
    batch = data_obj.to(device)
    z_dict = encoder(batch.x_dict, batch.edge_index_dict)
    edge_label_index, edge_label = sample_edge_labels(
        pos_edge_index,
        full_positive_edge_index,
        data_obj['user'].num_nodes,
        data_obj['item'].num_nodes,
        neg_ratio=1.0,
    )
    edge_label_index = edge_label_index.to(device)
    edge_label = edge_label.to(device)
    logits = decode_link_logits(z_dict, edge_label_index, decoder)
    y = edge_label.cpu().numpy()
    s = logits.detach().cpu().numpy()
    return {
        'n': int(y.size),
        'positives': int(y.sum()),
        'negatives': int((1 - y).sum()),
        'auroc': float(roc_auc_score(y, s)),
        'ap': float(average_precision_score(y, s)),
        'acc_0.5': float(((s >= 0) == y).mean()),
    }


def train_epochs_fullgraph(
    encoder: nn.Module,
    decoder: nn.Module,
    train_data: HeteroData,
    val_data: HeteroData,
    full_positive_edge_index: torch.Tensor,
    device: torch.device,
    epochs: int = 25,
    lr: float = 5e-4,
    wd: float = 1e-5,
    patience: int = 5,
):
    params = list(encoder.parameters()) + list(decoder.parameters())
    opt = torch.optim.Adam(params, lr=lr, weight_decay=wd)
    loss_fn = nn.BCEWithLogitsLoss()
    history = []
    best_val_auroc = float('-inf')
    best_epoch = 0
    best_encoder_state = None
    best_decoder_state = None
    wait = 0

    for epoch in range(1, epochs + 1):
        encoder.train()
        decoder.train()
        batch = train_data.to(device)
        opt.zero_grad()
        z_dict = encoder(batch.x_dict, batch.edge_index_dict)
        edge_label_index, edge_label = sample_edge_labels(
            train_data['user', 'reviews', 'item'].edge_label_index,
            full_positive_edge_index,
            train_data['user'].num_nodes,
            train_data['item'].num_nodes,
            neg_ratio=1.0,
        )
        edge_label_index = edge_label_index.to(device)
        edge_label = edge_label.to(device)
        logits = decode_link_logits(z_dict, edge_label_index, decoder)
        loss = loss_fn(logits, edge_label)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(params, max_norm=1.0)
        opt.step()

        val_metrics = eval_fullgraph(
            encoder,
            decoder,
            val_data,
            val_data['user', 'reviews', 'item'].edge_label_index,
            full_positive_edge_index,
            device,
        )
        history.append(
            {
                'epoch': epoch,
                'train_loss': float(loss.item()),
                'val_auroc': float(val_metrics['auroc']),
                'val_ap': float(val_metrics['ap']),
                'val_acc_0.5': float(val_metrics['acc_0.5']),
            }
        )
        print(
            'Epoch {:02d} train_loss={:.4f} val_AUROC={:.4f} val_AP={:.4f} val_acc@0.5={:.4f}'.format(
                epoch,
                loss.item(),
                val_metrics['auroc'],
                val_metrics['ap'],
                val_metrics['acc_0.5'],
            )
        )

        if val_metrics['auroc'] > best_val_auroc:
            best_val_auroc = float(val_metrics['auroc'])
            best_epoch = epoch
            best_encoder_state = copy.deepcopy(encoder.state_dict())
            best_decoder_state = copy.deepcopy(decoder.state_dict())
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print(f'Early stopping at epoch {epoch:02d}; best epoch was {best_epoch:02d}.')
                break

    if best_encoder_state is not None:
        encoder.load_state_dict(best_encoder_state)
        decoder.load_state_dict(best_decoder_state)

    return history


device: cuda
torch: 2.6.0+cu124


## Part A - Seen-User Full-Graph Training

This section mirrors notebook 06 in terms of data split and decoder contract, but trains one full-graph GCN forward per epoch instead of neighborhood mini-batches.

In [2]:
# Load the same hetero graph artifact used by notebook 06.
# For GCN, symmetrizing the item-item relation gives a more faithful normalized graph.
data = torch.load(PROCESSED_DIR / 'hetero_data.pt', weights_only=False)
data = make_item_graph_undirected(data)
full_seen_positive_edges = data['user', 'reviews', 'item'].edge_index.clone()
print(data)

# Keep the same edge-level split style as the GraphSAGE notebook.
transform = RandomLinkSplit(
    num_val=0.1,
    num_test=0.1,
    disjoint_train_ratio=0.2,
    neg_sampling_ratio=0.0,
    add_negative_train_samples=False,
    edge_types=('user', 'reviews', 'item'),
    rev_edge_types=None,
)
train_data, val_data, test_data = transform(data)
train_data = add_reverse_reviews(train_data)
val_data = add_reverse_reviews(val_data)
test_data = add_reverse_reviews(test_data)
metadata = train_data.metadata()

IN_CHANNELS = data['user'].x.size(1)
HIDDEN = 256
OUT_CH = 256
DROPOUT = 0.1
EPOCHS = 25
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 1e-5
EARLY_STOPPING_PATIENCE = 5

report = {
    'status': 'initialized',
    'seed': SEED,
    'device': str(device),
    'input_artifacts': {
        'hetero_data': str(PROCESSED_DIR / 'hetero_data.pt'),
        'hidden_interactions': str(PROCESSED_DIR / 'hidden_interactions_test.parquet'),
        'user_mapping': str(PROCESSED_DIR / 'user_mapping.json'),
        'item_mapping': str(PROCESSED_DIR / 'item_mapping.json'),
    },
    'output_artifact': str(CKPT_PATH),
    'schema': {
        'node_types': list(metadata[0]),
        'edge_types': [list(edge_type) for edge_type in metadata[1]],
    },
    'hparams': {
        'in_channels': IN_CHANNELS,
        'hidden_channels': HIDDEN,
        'out_channels': OUT_CH,
        'dropout': DROPOUT,
        'epochs': EPOCHS,
        'learning_rate': LEARNING_RATE,
        'weight_decay': WEIGHT_DECAY,
        'early_stopping_patience': EARLY_STOPPING_PATIENCE,
        'training_mode': 'full_graph',
        'item_graph_symmetrized': True,
    },
}

print('training mode:', report['hparams']['training_mode'])


HeteroData(
  user={ x=[1642, 384] },
  item={ x=[21639, 384] },
  (user, reviews, item)={ edge_index=[2, 10751] },
  (item, also_bought, item)={ edge_index=[2, 14506] }
)
training mode: full_graph


In [3]:
# First, record the raw limitation of native GCNConv on bipartite hetero edges.
def run_native_gcn_smoke():
    native_encoder = build_native_gcn_encoder(metadata, hidden_channels=HIDDEN, out_channels=OUT_CH, dropout=DROPOUT).to(device)
    batch = train_data.to(device)
    z_dict = native_encoder(batch.x_dict, batch.edge_index_dict)
    return {
        'user_embedding_shape': list(z_dict['user'].shape),
        'item_embedding_shape': list(z_dict['item'].shape),
    }


report['native_gcn_smoke'] = attempt_stage('native_gcn_smoke', run_native_gcn_smoke)
print_stage_result(report['native_gcn_smoke'])

# Then switch to the adapted encoder so the baseline can actually train and be evaluated.
encoder = build_adapted_gcn_encoder(metadata, hidden_channels=HIDDEN, out_channels=OUT_CH, dropout=DROPOUT).to(device)
decoder = DotProductDecoder().to(device)


def seen_user_sanity():
    batch = train_data.to(device)
    z_dict = encoder(batch.x_dict, batch.edge_index_dict)
    edge_label_index, _ = sample_edge_labels(
        train_data['user', 'reviews', 'item'].edge_label_index,
        full_seen_positive_edges,
        train_data['user'].num_nodes,
        train_data['item'].num_nodes,
        neg_ratio=1.0,
    )
    logits = decode_link_logits(z_dict, edge_label_index.to(device), decoder)
    return {
        'user_embedding_shape': list(z_dict['user'].shape),
        'item_embedding_shape': list(z_dict['item'].shape),
        'logits_shape': list(logits.shape),
    }


report['seen_user_sanity'] = attempt_stage('seen_user_sanity', seen_user_sanity)
print_stage_result(report['seen_user_sanity'])

if report['seen_user_sanity']['ok']:
    report['seen_user_training'] = attempt_stage(
        'seen_user_training',
        lambda: train_epochs_fullgraph(
            encoder,
            decoder,
            train_data,
            val_data,
            full_seen_positive_edges,
            device,
            epochs=EPOCHS,
            lr=LEARNING_RATE,
            wd=WEIGHT_DECAY,
            patience=EARLY_STOPPING_PATIENCE,
        ),
    )
    if report['seen_user_training']['ok']:
        report['seen_user_test'] = attempt_stage(
            'seen_user_test',
            lambda: eval_fullgraph(
                encoder,
                decoder,
                test_data,
                test_data['user', 'reviews', 'item'].edge_label_index,
                full_seen_positive_edges,
                device,
            ),
        )
        print_stage_result(report['seen_user_test'])
        report['status'] = 'trained'
    else:
        report['seen_user_test'] = None
        report['status'] = 'blocked_during_training'
else:
    report['seen_user_training'] = None
    report['seen_user_test'] = None
    report['status'] = 'blocked_on_seen_user_forward'

# Save an intermediate report before moving on to the hidden-user evaluation.
torch.save(report, CKPT_PATH)
print('Saved partial report:', CKPT_PATH.name)


[native_gcn_smoke] failed: ValueError - 'GCNConv' received a tuple of node features as input while this layer does not support bipartite message passing. Please try other layers such as 'SAGEConv' or 'GraphConv' instead
[seen_user_sanity] ok
  user_embedding_shape: [1642, 256]
  item_embedding_shape: [21639, 256]
  logits_shape: [3440]
Epoch 01 train_loss=0.6928 val_AUROC=0.9618 val_AP=0.9485 val_acc@0.5=0.8526
Epoch 02 train_loss=0.6829 val_AUROC=0.9748 val_AP=0.9643 val_acc@0.5=0.9512
Epoch 03 train_loss=0.6726 val_AUROC=0.9716 val_AP=0.9556 val_acc@0.5=0.9540
Epoch 04 train_loss=0.6603 val_AUROC=0.9732 val_AP=0.9615 val_acc@0.5=0.9563
Epoch 05 train_loss=0.6450 val_AUROC=0.9724 val_AP=0.9583 val_acc@0.5=0.9535
Epoch 06 train_loss=0.6268 val_AUROC=0.9766 val_AP=0.9670 val_acc@0.5=0.9605
Epoch 07 train_loss=0.6014 val_AUROC=0.9725 val_AP=0.9570 val_acc@0.5=0.9577
Epoch 08 train_loss=0.5727 val_AUROC=0.9711 val_AP=0.9522 val_acc@0.5=0.9586
Epoch 09 train_loss=0.5403 val_AUROC=0.9754 va

## Part B - Hidden-User Full-Graph Evaluation

This section rebuilds the holdout-user graph from `hidden_interactions_test.parquet`, reuses the trained encoder/decoder, and compares hidden-user quality against the seen-user full-graph split.

In [4]:
# Rebuild the inductive graph for users that were held out in notebook 01.
hidden_path = PROCESSED_DIR / 'hidden_interactions_test.parquet'
if not hidden_path.is_file():
    raise FileNotFoundError(f'Missing {hidden_path}. Run notebook 01 first.')

with open(PROCESSED_DIR / 'user_mapping.json') as f:
    seen_user_ids = set(json.load(f).keys())

with open(PROCESSED_DIR / 'item_mapping.json') as f:
    item_mapping = json.load(f)

hdf = pd.read_parquet(hidden_path)
x_item_full = data['item'].x.clone()
ii_global = data['item', 'also_bought', 'item'].edge_index.clone()

hidden_users = hdf['reviewerID'].unique().tolist()
hidden_user_map = {uid: idx for idx, uid in enumerate(hidden_users)}

rows = []
dropped_oov = 0
for _, row in hdf.iterrows():
    if row['reviewerID'] in seen_user_ids:
        raise ValueError(f"Hidden user leaked into seen set: {row['reviewerID']}")
    asin = row['asin']
    if asin not in item_mapping:
        dropped_oov += 1
        continue
    rows.append((hidden_user_map[row['reviewerID']], int(item_mapping[asin])))

if not rows:
    raise ValueError('No inductive edges available after filtering OOV items.')

ui_ind = torch.tensor(rows, dtype=torch.long).t().contiguous()
data_ind = HeteroData()
data_ind['user'].x = torch.zeros(len(hidden_user_map), IN_CHANNELS, dtype=torch.float)
data_ind['item'].x = x_item_full
data_ind['user', 'reviews', 'item'].edge_index = ui_ind
data_ind['item', 'also_bought', 'item'].edge_index = ii_global
full_inductive_positive_edges = data_ind['user', 'reviews', 'item'].edge_index.clone()

ind_transform = RandomLinkSplit(
    num_val=0.1,
    num_test=0.1,
    disjoint_train_ratio=0.2,
    neg_sampling_ratio=0.0,
    add_negative_train_samples=False,
    edge_types=('user', 'reviews', 'item'),
    rev_edge_types=None,
)
ind_train, ind_val, ind_test = ind_transform(data_ind)
ind_train = add_reverse_reviews(ind_train)
ind_val = add_reverse_reviews(ind_val)
ind_test = add_reverse_reviews(ind_test)

assert set(ind_train.node_types) == set(metadata[0])
assert set(ind_train.edge_types) == set(metadata[1])

report['inductive_graph'] = {
    'hidden_users': int(len(hidden_user_map)),
    'kept_edges': int(ui_ind.size(1)),
    'dropped_oov_asin': int(dropped_oov),
}


def inductive_hidden_user_sanity():
    batch = ind_test.to(device)
    z_dict = encoder(batch.x_dict, batch.edge_index_dict)
    edge_label_index, _ = sample_edge_labels(
        ind_test['user', 'reviews', 'item'].edge_label_index,
        full_inductive_positive_edges,
        ind_test['user'].num_nodes,
        ind_test['item'].num_nodes,
        neg_ratio=1.0,
    )
    logits = decode_link_logits(z_dict, edge_label_index.to(device), decoder)
    return {
        'user_embedding_shape': list(z_dict['user'].shape),
        'item_embedding_shape': list(z_dict['item'].shape),
        'logits_shape': list(logits.shape),
    }


report['inductive_hidden_user_sanity'] = attempt_stage(
    'inductive_hidden_user_sanity',
    inductive_hidden_user_sanity,
)
print_stage_result(report['inductive_hidden_user_sanity'])

if report['status'] == 'trained' and report['inductive_hidden_user_sanity']['ok']:
    report['inductive_hidden_user_test'] = attempt_stage(
        'inductive_hidden_user_test',
        lambda: eval_fullgraph(
            encoder,
            decoder,
            ind_test,
            ind_test['user', 'reviews', 'item'].edge_label_index,
            full_inductive_positive_edges,
            device,
        ),
    )
    print_stage_result(report['inductive_hidden_user_test'])
else:
    report['inductive_hidden_user_test'] = None

random_edge_label_index, random_edge_label = sample_edge_labels(
    ind_test['user', 'reviews', 'item'].edge_label_index,
    full_inductive_positive_edges,
    ind_test['user'].num_nodes,
    ind_test['item'].num_nodes,
    neg_ratio=1.0,
)
random_scores = np.random.default_rng(SEED).standard_normal(random_edge_label.shape[0])
report['inductive_baseline_random_auroc'] = float(
    roc_auc_score(random_edge_label.numpy(), random_scores)
)

seen_metrics = report.get('seen_user_test')
ind_metrics = report.get('inductive_hidden_user_test')
if seen_metrics and seen_metrics.get('ok') and ind_metrics and ind_metrics.get('ok'):
    seen_auroc = float(seen_metrics['value']['auroc'])
    ind_auroc = float(ind_metrics['value']['auroc'])
    report['inductive_gap_auroc'] = float(seen_auroc - ind_auroc)
else:
    report['inductive_gap_auroc'] = None

if report['native_gcn_smoke']['ok']:
    report['limitations_summary'] = 'Native GCNConv passed the bipartite smoke-test in this environment.'
else:
    native_error = report['native_gcn_smoke']['error']['message']
    if 'tuple of node features' in native_error and report['inductive_gap_auroc'] is not None:
        report['limitations_summary'] = (
            'Raw GCNConv inside to_hetero fails on bipartite tuple inputs; the adapted encoder runs, '
            'but hidden-user AUROC is lower than seen-user AUROC by {:.4f}.'.format(
                report['inductive_gap_auroc']
            )
        )
    elif 'tuple of node features' in native_error:
        report['limitations_summary'] = (
            'Raw GCNConv inside to_hetero fails on bipartite tuple inputs. The adapted encoder is used '
            'for the runnable baseline saved in this checkpoint.'
        )
    else:
        report['limitations_summary'] = native_error

# Final checkpoint: weights + metrics + limitation notes for the report.
torch.save(
    {
        **report,
        'encoder': encoder.state_dict(),
        'decoder': decoder.state_dict(),
    },
    CKPT_PATH,
)

summary = {
    'status': report['status'],
    'training_mode': report['hparams']['training_mode'],
    'limitations_summary': report['limitations_summary'],
}
print('final summary:', summary)
print('Saved final report:', CKPT_PATH.name)


[inductive_hidden_user_sanity] ok
  user_embedding_shape: [183, 256]
  item_embedding_shape: [21639, 256]
  logits_shape: [240]
[inductive_hidden_user_test] ok
  auroc: 0.8391666666666666
  ap: 0.8803804688698029
  acc_0.5: 0.7583333333333333
final summary: {'status': 'trained', 'training_mode': 'full_graph', 'limitations_summary': 'Raw GCNConv inside to_hetero fails on bipartite tuple inputs; the adapted encoder runs, but hidden-user AUROC is lower than seen-user AUROC by 0.1470.'}
Saved final report: gcn_link_pred.pt
